In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Initial Data Exploration
* Load the dataset using Pandas. Check for null values and understand data types.
* Examine the time series properties of the data (e.g., frequency, trends).

In [ ]:
apple_stock = pd.read_csv("Apple Stock Prices (1981 to 2023).csv")

print(apple_stock.head())
print(apple_stock.info())

In [ ]:
apple_stock['Date'] = pd.to_datetime(apple_stock['Date'], format='%d/%m/%Y')
apple_stock.set_index('Date', inplace=True)

In [ ]:
apple_stock.isnull().sum()

In [ ]:
# Check if the data follows any consistent pattern (e.g., daily business days)

# Check the inferred frequency of the index
inferred_freq = pd.infer_freq(apple_stock.index)
print(f"Inferred Frequency: {inferred_freq}")

# Check for missing business days (B = business day)
full_range = pd.date_range(start=apple_stock.index.min(), end=apple_stock.index.max(), freq='B')
missing_dates = full_range.difference(apple_stock.index)
print(f"Number of missing business days: {missing_dates}")

In [ ]:
#Evaluate with Simple Moving Averages (SMA)

apple_stock['SMA_20'] = apple_stock['Adj Close'].rolling(window=20).mean()
apple_stock['SMA_50'] = apple_stock['Adj Close'].rolling(window=50).mean()

In [ ]:
monthly_trend = apple_stock['Adj Close'].resample('ME').mean()

In [ ]:
apple_stock_modern = apple_stock.loc['2000':].copy()
apple_stock_modern['SMA_200'] = apple_stock_modern['Adj Close'].rolling(window=200).mean()
apple_stock_modern['SMA_50'] = apple_stock_modern['Adj Close'].rolling(window=50).mean()
monthly_trend_modern = apple_stock_modern['Adj Close'].resample('ME').mean()

# Visualize SME and monthly trend 
plt.figure(figsize=(14,7))

# Plot daily price
plt.plot(apple_stock_modern.index, apple_stock_modern['Adj Close'], label='Daily Adj Close', alpha=0.3, color='gray')

# Plot SMA
plt.plot(apple_stock_modern.index, apple_stock_modern['SMA_50'], label='50 Day SMA', color='blue')
plt.plot(apple_stock_modern.index, apple_stock_modern['SMA_200'], label='200 Day SMA', color='orange', linewidth=2)


# Plot monthly trend
monthly_avg = apple_stock_modern['Adj Close'].resample('ME').mean()
plt.plot(monthly_avg.index, monthly_avg, label='Monthly Average', color='red', alpha=0.6)

plt.title('Apple Stock Trends (2000 - Present)', fontsize=16)
plt.yscale('log')
plt.ylabel('Price (Log Scale)')
plt.legend()
plt.grid(True, which='both', alpha=0.2)
plt.show()

### Data Visualization
* Utilize Matplotlib to plot closing prices and traded volume over time.
* Create a candlestick chart to depict high and low prices.

In [ ]:
# CReate a figure with two subplots: Price and Volume
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(14,10), sharex=True, gridspec_kw={'height_ratios': [3,1]})

# Plot 1: Closing Prices
ax1.plot(apple_stock_modern.index, apple_stock_modern['Adj Close'], label='Adj Close', color='blue')
ax1.set_title('Apple Stock Price and Volume (2000 - Present)')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Traded Volume
ax2.bar(apple_stock_modern.index, apple_stock_modern['Volume'], color='grey', alpha=0.5)
ax2.set_ylabel('Volume (Shares)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import os, sys
os.system(f"{sys.executable} -m pip install mplfinance")

In [ ]:
import mplfinance as mpf

In [ ]:
recent_data = apple_stock.tail(50)

# Create candlestick chart 
mpf.plot(recent_data, type='candle', style='charles',
         title='Apple Candlestick Chart (Recent 50 Days)',
         ylabel='Price ($)',
         volume=True,
         ylabel_lower='Volume', 
         mav=(10),
         figsize=(14,8))

### Statistical Analysis
* Compute summary statistics (mean, median, standard deviation) for key columns.
* Analyze closing prices with a moving average.


In [ ]:
# Summary Statistics
stats = apple_stock.describe()

median_close = apple_stock['Adj Close'].median()
std_close = apple_stock['Adj Close'].std()

print(stats)
print(f"\nMedian Adj Close: {median_close:.2f}")
print(f"Standard Deviation: {std_close:.2f}")

In [ ]:
# Short term trend
apple_stock['SMA_50'] = apple_stock['Adj Close'].rolling(window=50).mean()

# Long term trend
apple_stock['SMA_200'] = apple_stock['Adj Close'].rolling(window=200).mean()

# Calculate spread
apple_stock['SMA_Delta'] = apple_stock['SMA_50'] - apple_stock['SMA_200']

# Identify the Golden Cross and Dealth Cross

golden_cross_days = apple_stock[(apple_stock["SMA_50"] > apple_stock['SMA_200']) & 
                                (apple_stock['SMA_50'].shift(1) <= apple_stock['SMA_200'].shift(1))]

print("Recent Golden Cross dates:")
print(golden_cross_days.tail())

### Hypothesis Testing
* Execute a t-test to compare average closing prices across different years.
* Examine daily returns’ distribution and test for normality using SciPy.


In [ ]:
from scipy import stats

In [ ]:
# T-Test

prices_2022 = apple_stock_modern.loc['2022', 'Adj Close']
prices_2023 = apple_stock_modern.loc['2023', 'Adj Close']

t_stat, p_val = stats.ttest_ind(prices_2022, prices_2023)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_val:.4f}")

if p_val < -0.05:
    print("Result: Significant difference in the average prices between 2022 and 2023.")
else: 
    print("Result: No significant difference found")

In [ ]:
# Normality test

apple_stock_modern['Returns'] = apple_stock_modern['Adj Close'].pct_change().dropna()
returns = apple_stock_modern['Returns'].dropna()

stat, p_val_norm = stats.normaltest(returns)

print(f"Normality Test P-Value: {p_val_norm:.4f}")

if p_val_norm < 0.05:
    print("Result: The returns are not normally distributed")
else:
    print("Result: The returns appear to follow a normal distribution")

### Advanced Statistical Techniques (Bonus)
* Statistical Functions in NumPy: Employ NumPy’s statistical functions for in-depth stock data analysis.
    * E.g., Use convolve for moving averages, or np.corrcoef to explore correlations between financial metrics.
    * Analyze correlations between moving averages of closing prices and trading volume across time periods.